# Fraud Detection - Model Training (SPCS)

This notebook trains an XGBoost fraud detection model using **Snowflake ML Jobs** on **Snowpark Container Services (SPCS)**.

**Why SPCS over Warehouse?**
- Container-based compute with flexible resource allocation
- Support for custom Python environments and dependencies
- Better suited for long-running, resource-intensive training jobs
- GPU support available for deep learning workloads

**Workflow:**
1. Submit training job via `submit_directory()` to SPCS compute pool
2. Training script (`source/train.py`) loads data from Feature Store snapshot
3. Model artifacts saved to Snowflake stage
4. Register trained model to Snowflake Model Registry

### Import Libraries

- `submit_directory`: Submits a local directory as an ML Job to SPCS
- `Registry`: Interface to Snowflake Model Registry for model versioning
- `type_hints`: Specifies target deployment platform (SPCS vs Warehouse)

In [ ]:
import os
import pickle
import time
import snowflake.connector
from snowflake.snowpark import Session
from snowflake.ml.jobs import submit_directory
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    conn = snowflake.connector.connect(
        connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME")
    )
    session = Session.builder.configs({"connection": conn}).create()

print(f"Connected: {session.get_current_account()}")

### Configuration

Set up database context and define where model artifacts will be stored. The `ML_MODELS` stage holds serialized model files and metrics.

In [ ]:
DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
WAREHOUSE = "ML_WH"
MODEL_NAME = "FRAUD_DETECTION_MODEL"
MODEL_STAGE = f"@{DATABASE}.{SCHEMA}.ML_MODELS"

session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
print(f"Context: {DATABASE}.{SCHEMA} | Warehouse: {WAREHOUSE}")

### Create Compute Pool (if needed)

SPCS requires a **compute pool** - a managed cluster of containers that execute ML jobs.

- `CPU_X64_S`: Small CPU instance (suitable for XGBoost)
- `AUTO_SUSPEND_SECS=300`: Pool suspends after 5 minutes of inactivity to save costs
- For GPU workloads, use `GPU_NV_S` or similar instance families

In [ ]:
COMPUTE_POOL = "ML_TRAINING_POOL"

pool_exists = session.sql(f"SHOW COMPUTE POOLS LIKE '{COMPUTE_POOL}'").collect()
if not pool_exists:
    session.sql(f"""
        CREATE COMPUTE POOL {COMPUTE_POOL}
        MIN_NODES = 1 MAX_NODES = 1
        INSTANCE_FAMILY = CPU_X64_S
        AUTO_SUSPEND_SECS = 300
    """).collect()
    print(f"Created compute pool: {COMPUTE_POOL}")
else:
    print(f"Compute pool exists: {COMPUTE_POOL}")

## Submit Training Job

`submit_directory()` packages the `source/` directory and runs `train.py` on SPCS:

- **dir_path**: Local directory containing training code (`train.py` + shared `utils.py`)
- **entrypoint**: Script to execute (`train.py`)
- **compute_pool**: SPCS pool to run the job
- **stage_name**: Snowflake stage for job artifacts
- **env_vars**: Environment variables passed to the training script

The training script:
1. Loads `FRAUD_TRAINING_DATA` (generated from Feature Store) and validates it has every column in `FEATURE_COLS`
2. Splits **chronologically**: oldest 64% train, next 16% validation, most recent 20% test
3. Trains XGBoost with class imbalance handling (`scale_pos_weight`)
4. Picks the APPROVE/REVIEW/DECLINE thresholds on the validation slice
5. Reports AUC-ROC, AUC-PR, recall at 1%/5% FPR and precision/recall/F1 at the chosen threshold on the test slice
6. Uploads model (with thresholds), metrics and feature importance (gain + mean |SHAP|) to the stage and appends run-keyed predictions to `FRAUD_TEST_DATA`

In [ ]:
EXPERIMENT_NAME = "FRAUD_DETECTION_EXPERIMENT"

job = submit_directory(
    dir_path="../source",
    entrypoint="train.py",
    compute_pool=COMPUTE_POOL,
    stage_name=f"{DATABASE}.{SCHEMA}.ML_JOB_STAGE",
    env_vars={
        "DATABASE": DATABASE,
        "SCHEMA": SCHEMA,
        "MODEL_STAGE": MODEL_STAGE,
        "EXPERIMENT_NAME": EXPERIMENT_NAME,
    }
)
print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")

### Monitor Job Progress

`job.wait()` blocks until the job completes. Job logs stream training output including:
- Data loading progress and the time range of each split
- Validation-selected decision thresholds
- Evaluation metrics (AUC-PR, recall@FPR, precision/recall/F1 at the threshold, AUC-ROC)
- Top features by mean |SHAP|
- Artifact upload confirmation

In [ ]:
print("Waiting for job to complete...")
job.wait()

print(f"\nFinal status: {job.status}")
if job.status == "DONE":
    print("\n=== Job Logs ===")
    logs = job.get_logs()
    print(logs[-3000:] if len(logs) > 3000 else logs)
else:
    print(f"Job failed: {job.status}")
    print(job.get_logs())

## Register Model

Download trained artifacts from stage:
- `fraud_model.pkl`: Serialized XGBoost model + feature list + decision thresholds
- `metrics.pkl`: Evaluation metrics dictionary
- `feature_importance.json`: Gain and mean |SHAP| per feature

These will be used to register the model with full lineage tracking.

In [ ]:
import json

session.sql(f"GET {MODEL_STAGE}/fraud_model.pkl file:///tmp/").collect()
session.sql(f"GET {MODEL_STAGE}/metrics.pkl file:///tmp/").collect()
session.sql(f"GET {MODEL_STAGE}/feature_importance.json file:///tmp/").collect()

with open("/tmp/fraud_model.pkl", "rb") as f:
    model_data = pickle.load(f)
    model = model_data["model"]
    feature_cols = model_data["features"]
    thresholds = model_data["decision_thresholds"]

with open("/tmp/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)

with open("/tmp/feature_importance.json") as f:
    importance = json.load(f)

print(f"Loaded model with {len(feature_cols)} features")
print(f"Decision bands: APPROVE < {thresholds['review_threshold']:.4f} <= REVIEW < {thresholds['decline_threshold']:.4f} <= DECLINE")
print(f"\nMetrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
print("\nTop features (mean |SHAP|):")
for row in importance[:10]:
    print(f"  {row['feature']}: {row['mean_abs_shap']:.4f}")

### Log to Model Registry

The Model Registry provides:
- **Version control**: Track multiple model versions (V1_SPCS, V2_TUNED, etc.)
- **Lineage**: Automatic tracking of training data sources
- **Metrics storage**: Attach evaluation metrics to model versions
- **Deployment targets**: `SNOWPARK_CONTAINER_SERVICES` enables SPCS inference services

`sample_input_data` defines the expected input schema for inference.

In [ ]:
import pandas as pd

registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

sample_df = session.table(f"{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA").select(feature_cols).limit(100)

VERSION_NAME = "V3_RETRAIN"
registry_metrics = {k: v for k, v in metrics.items() if isinstance(v, (int, float, str))}
registry_metrics["auc"] = metrics["auc_roc"]

try:
    mv = registry.log_model(
        model=model,
        model_name=MODEL_NAME,
        version_name=VERSION_NAME,
        sample_input_data=sample_df,
        comment="XGBoost fraud detection - point-in-time features, time-based evaluation",
        metrics=registry_metrics,
        target_platforms=[type_hints.TargetPlatform.SNOWPARK_CONTAINER_SERVICES]
    )
    print(f"\nModel registered: {MODEL_NAME} {VERSION_NAME}")
except ValueError as e:
    if "already existed" in str(e):
        print(f"Model {MODEL_NAME} {VERSION_NAME} already exists - skipping registration")
        mv = registry.get_model(MODEL_NAME).version(VERSION_NAME)
    else:
        raise
print(f"Location: {DATABASE}.{SCHEMA}")

In [ ]:
print("\n=== Training Summary ===")
print(f"Compute: SPCS ({COMPUTE_POOL})")
print(f"Model: {MODEL_NAME} {VERSION_NAME}")
print(f"Registry: {DATABASE}.{SCHEMA}")
print(f"\nKey Metrics (most recent 20% of transactions):")
print(f"  AUC-PR:        {metrics.get('auc_pr', 0):.4f}")
print(f"  Recall@1%FPR:  {metrics.get('recall_at_1pct_fpr', 0):.4f}")
print(f"  Recall@5%FPR:  {metrics.get('recall_at_5pct_fpr', 0):.4f}")
print(f"  Precision/Recall/F1 @ {metrics.get('decline_threshold', 0):.3f}: "
      f"{metrics.get('precision', 0):.4f} / {metrics.get('recall', 0):.4f} / {metrics.get('f1', 0):.4f}")
print(f"  AUC-ROC:       {metrics.get('auc_roc', 0):.4f}")
print(f"  Baseline RISK_SCORE AUC-PR: {metrics.get('baseline_risk_score_auc_pr', 0):.4f}")
print(f"\n Proceed to 03b_model_tune.ipynb for hyperparameter tuning")